In [9]:
!pip install transformers datasets scikit-learn pandas torch accelerate -q

In [10]:
import pandas as pd

df = pd.read_csv('Resume.csv')
print(df.shape)
print(df['Category'].value_counts())
df.head()

(2484, 4)
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])

print(f"Total classes: {len(le.classes_)}")
print(le.classes_)

Total classes: 24
['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


In [13]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 1987 | Test: 497


In [27]:
from transformers import DistilBertTokenizer
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(texts, max_len=256):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

train_encodings = tokenize(train_df['Resume_str'])
test_encodings  = tokenize(test_df['Resume_str'])

In [28]:
class ResumeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = ResumeDataset(train_encodings, train_df['label'].values)
test_dataset  = ResumeDataset(test_encodings,  test_df['label'].values)

In [29]:
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

NUM_LABELS = len(le.classes_)

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=NUM_LABELS
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds)}

training_args = TrainingArguments(
    output_dir='./resume_model',
    num_train_epochs=4,              # increase epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=20,
    fp16=True,
    learning_rate=2e-5,             # add this - default is too high
    warmup_ratio=0.1,               # add this - stabilizes training
    weight_decay=0.01,              # add this - prevents overfitting
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,2.498953,2.227229,0.708249
2,1.570533,1.326691,0.774648
3,1.202670,1.070103,0.808853
4,0.988270,0.999452,0.818913


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=1.7287314262390137, metrics={'train_runtime': 178.0861, 'train_samples_per_second': 44.63, 'train_steps_per_second': 2.808, 'total_flos': 526631979368448.0, 'train_loss': 1.7287314262390137, 'epoch': 4.0})

In [30]:
results = trainer.evaluate()
print(f"\n✅ Test Accuracy: {results['eval_accuracy']*100:.2f}%")

Training Loss,Validation Loss,Epoch,Accuracy
0.988270,0.999452,4,0.818913



✅ Test Accuracy: 81.89%


In [31]:
# Save locally first
model.save_pretrained('./resume_classifier')
tokenizer.save_pretrained('./resume_classifier')

import json
label_map = {int(i): label for i, label in enumerate(le.classes_)}
with open('./resume_classifier/label_map.json', 'w') as f:
    json.dump(label_map, f)

print("✅ Model saved locally")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved locally


In [33]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

def predict_category(resume_text):
    inputs = tokenizer(
        resume_text,
        return_tensors='pt',
        truncation=True,
        max_length=256,
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}  # ← move to GPU
    with torch.no_grad():
        outputs = model(**inputs)
    pred_id = outputs.logits.argmax(-1).item()
    return label_map[pred_id]

sample = """
Experienced Python developer with 3 years in machine learning.
Built NLP pipelines using BERT, trained classification models,
deployed REST APIs with FastAPI. Familiar with TensorFlow and PyTorch.
"""

print(f"Predicted Category: {predict_category(sample)}")

Predicted Category: INFORMATION-TECHNOLOGY


In [34]:
import shutil
from google.colab import files

shutil.make_archive('resume_classifier', 'zip', './resume_classifier')
files.download('resume_classifier.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>